In [60]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

In [61]:
df=pd.read_csv("./Salary_Data.csv")
df=df.dropna()

In [62]:
y_train=df[["Salary"]].to_numpy()
X_train=df[["Age","Years of Experience"]].to_numpy()
y_train = y_train.flatten()
print(X_train)
print(y_train)



[[32.  5.]
 [28.  3.]
 [45. 15.]
 ...
 [30.  4.]
 [46. 14.]
 [26.  1.]]
[ 90000.  65000. 150000. ...  55000. 140000.  35000.]


In [63]:

# Phase 1: Core Algorithms
# ---------------------------------------------------------
def zscore_normalize_features(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

def compute_cost(X, y, w, b):
    m = X.shape[0]
    predictions = np.dot(X, w) + b
    errors = predictions - y
    cost = (1 / (2 * m)) * np.sum(errors ** 2)
    return cost

def gradient_descent(X, y, w_in, b_in, alpha, num_iters):
    w = w_in.copy()

    b = b_in
    m = X.shape[0]
    j_hist = []
    
    for i in range(num_iters):
        predictions = np.dot(X, w) + b
        errors = predictions - y
        
        dj_dw = (1 / m) * np.dot(X.T, errors)
        dj_db = (1 / m) * np.sum(errors)
        
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        
        if i % (num_iters // 10) == 0 or i == num_iters - 1:
            current_cost = compute_cost(X, y, w, b)
            j_hist.append(current_cost)
            print(f"Iteration {i:4d}: Cost {current_cost:.2f}")
            
    return w, b, j_hist

In [75]:
print("Engineering and Scaling Features...")

sqr_age=(X_train[:,0]**2).reshape(-1,1)
X_engineered=np.hstack((X_train,sqr_age))
scaler=StandardScaler()
X_scaled_sklearn=scaler.fit_transform(X_engineered)
print(scalar)

X_engineered_df=pd.DataFrame(X_engineered, 
    columns=["Age", "Years of Experience", "Age_Squared"])
print(X_engineered_df)

Engineering and Scaling Features...
StandardScaler()
       Age  Years of Experience  Age_Squared
0     32.0                  5.0       1024.0
1     28.0                  3.0        784.0
2     45.0                 15.0       2025.0
3     36.0                  7.0       1296.0
4     52.0                 20.0       2704.0
...    ...                  ...          ...
6693  49.0                 20.0       2401.0
6694  32.0                  3.0       1024.0
6695  30.0                  4.0        900.0
6696  46.0                 14.0       2116.0
6697  26.0                  1.0        676.0

[6698 rows x 3 columns]


In [76]:
X_scaled, mu, sigma = zscore_normalize_features(X_engineered)

In [77]:
print(X_scaled)

[[-0.2131288  -0.51076899 -0.28929253]
 [-0.73839298 -0.84081078 -0.71136295]
 [ 1.49397978  1.13943997  1.47109285]
 ...
 [-0.47576089 -0.67578989 -0.50736224]
 [ 1.62529582  0.97441907  1.63112789]
 [-1.00102507 -1.17085257 -0.90129464]]


In [78]:
print("Initializing Gradient Descent...")
print("Initializing Gradient Descent...")
# We now have 3 features: [Experience, Age, Experience_Squared]
w_init = np.zeros(X_scaled.shape[1])
b_init = 0.0
alpha = 0.1
iterations = 1000
w_final, b_final, J_hist = gradient_descent(X_scaled, y_train, w_init, b_init, alpha, iterations)

print("\n--- Final Model Parameters ---")
print(f"Optimized Weights: {w_final}")
print(f"Optimized Bias: {b_final:.2f}")
def predict_salary(age, experience, w, b, mu, sigma):
    # 1. Engineer the exact same polynomial feature (Age Squared)
    age_squared = age ** 2
    
    # 2. Bundle the features in the exact same order as your X_train: [Age, Experience, Age_Squared]
    x_engineered = np.array([age, experience, age_squared])
    
    # 3. Scale using the historical mu and sigma generated during training
    x_scaled = (x_engineered - mu) / sigma
    
    # 4. Execute the mathematical prediction
    prediction = np.dot(x_scaled, w) + b
    
    return prediction

# ---------------------------------------------------------
# How to use it in your live system:
# ---------------------------------------------------------
print("\n--- Testing a New Applicant ---")
predicted_val = predict_salary(
    age=30, 
    experience=5, 
    w=w_final, 
    b=b_final, 
    mu=mu, 
    sigma=sigma
)
print(predicted_val)

Initializing Gradient Descent...
Initializing Gradient Descent...
Iteration    0: Cost 6379856179.14
Iteration  100: Cost 482681562.03
Iteration  200: Cost 463028797.17
Iteration  300: Cost 458453139.23
Iteration  400: Cost 456599493.15
Iteration  500: Cost 455343224.86
Iteration  600: Cost 454305642.80
Iteration  700: Cost 453407680.02
Iteration  800: Cost 452623140.34
Iteration  900: Cost 451936419.45
Iteration  999: Cost 451340729.36

--- Final Model Parameters ---
Optimized Weights: [ 19704.72435876  55540.09574238 -33769.79081832]
Optimized Bias: 115329.25

--- Testing a New Applicant ---
94719.87399751629


In [79]:
model=LinearRegression()
model.fit(X_scaled_sklearn,y_train)
x_single_applicant = np.array([[30, 5, 30**2]])

# Scale using the historical mu/sigma
x_single_scaled = scaler.transform(x_single_applicant)

# Predict the single value
single_prediction = model.predict(x_single_scaled)

# Extract the number from the output array using [0]
print(f"Predicted Salary: ${single_prediction[0]:,.2f}")

Predicted Salary: $95,630.45
